## Data Source and Setup

The temperature data used in this analysis comes from the NOAA NCEP Reanalysis
Daily Averages dataset. This dataset provides global atmospheric variables
derived from a combination of observations and numerical weather prediction
models.

We used the **air temperature at sigma level 995**
(`air.sig995`), which represents near-surface air temperature.

The analysis focuses on the period **1981–2000**. The following code initializes
the required libraries and defines the parameters needed to download and store
the dataset locally.


Dataset source: NOAA NCEP Reanalysis Daily Averages  
https://psl.noaa.gov/data/gridded/data.ncep.reanalysis.html


In [1]:
### Import Libraries
import os
import requests
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
import rioxarray as rio

BASE_URL = "https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis.dailyavgs/surface/"
DATA_DIR = "data"
VARIABLE = "air.sig995"
YEARS = range(1981, 2001)
os.makedirs(DATA_DIR, exist_ok=True)

---

## Downloading the NetCDF Files

The NOAA PSL server hosts the NCEP Reanalysis datasets as yearly NetCDF files.
The following functions are used to:

1. Identify the available NetCDF files for the selected variable and years.
2. Download the files locally if they are not already present.

This approach ensures reproducibility and avoids re-downloading files that
already exist in the local directory.


In [2]:
def get_netcdf_files(variable: str, year: int) -> pd.DataFrame:
    """ Fetch list of NetCDF files for a variable and year."""

    r = requests.get(BASE_URL)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    files = []
    for link in soup.find_all("a"):
        fname = link.get("href")

        if fname and fname.endswith(".nc"):
            if fname.startswith(variable) and f".{year}.nc" in fname:
                files.append({
                    "filename": fname,
                    "url": BASE_URL + fname
                })

    return pd.DataFrame(files)


def download_files(file_urls):
    
    for url in file_urls:
        filename = os.path.join(DATA_DIR, url.split("/")[-1])
        
        if os.path.exists(filename):
            print(f"{filename} Exists")
            continue
        print(f"Downloading: {filename}")
        
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
        
            with open(filename, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)


def save_index_nc(dataarray, filename, out_path="output", units="degC"):
    """
    Save an xarray DataArray to NetCDF with updated metadata.
    """
    # create folder
    os.makedirs(out_path, exist_ok=True)

    encoding = {
    dataarray.name: {
        "zlib": True,
        "complevel": 4
    }
}
    # save file
    out_file = os.path.join(out_path, filename)
    dataarray.to_netcdf(out_file, engine="netcdf4", encoding = encoding)
    return out_file

In [ ]:
# dfs = [get_netcdf_files(VARIABLE, y) for y in YEARS]
# files_df = pd.concat(dfs, ignore_index=True)


# download_files(files_df["url"].tolist())


KeyboardInterrupt: 

---

## Data Processing and Aggregation

After loading the dataset, One preprocessing steps was performed before analysis.

The air temperature variable is converted from **Kelvin to Celsius**, which is a more interpretable unit for temperature analysis.

Finally, the dataset is aggregated to compute different temporal summaries:

- **Monthly mean temperature** between 1981 and 2000
- **Annual mean temperature** between 1981 and 2000
- **Summer seasonal mean temperature**, calculated using data from June, July, and August

These aggregated datasets are then exported as NetCDF files for further analysis and visualization.

In [ ]:
# LOAD DATA WITH XARRAY
ds = xr.open_mfdataset(
    f"{DATA_DIR}/{VARIABLE}.*.nc",
    combine="by_coords",
    engine="netcdf4",
    chunks={"time": 365}
)

# Convert longitude 0–360 → -180–180
ds = ds.assign_coords(
    lon=((ds.lon + 180) % 360) - 180
)

ds = ds.sortby("lon")

# Update longitude metadata
ds.lon.attrs["actual_range"] = [
    float(ds.lon.min()),
    float(ds.lon.max())
]

# Convert to Celsius
ds["air"] = ds["air"] - 273.15
ds.air.attrs["units"] = "degC"
ds.air.attrs.pop("valid_range", None)

# Compute once
air_temp_celsius = ds.air.compute()

out_path = "output/"

# Monthly Mean
monthly_avg = air_temp_celsius.resample(time="1MS").mean()
save_index_nc(monthly_avg, "monthly_average.nc", out_path)

# Annual Mean
yearly = air_temp_celsius.resample(time="1YS").mean()
save_index_nc(yearly, "annual_mean_1981-2000.nc", out_path)

# Summer Seasonal Mean
summer = air_temp_celsius.sel(time=air_temp_celsius.time.dt.month.isin([6, 7, 8]))
summer_mean = summer.groupby("time.year").mean()
save_index_nc(summer_mean, "summer_seasonal_mean.nc", out_path)

---

## Mean Temperature Map – December 1992

To visualize the spatial distribution of temperature, the mean air
temperature for **December 1992** is extracted from the aggregated dataset.

This map illustrates how temperature varies geographically during the winter
season in the Northern Hemisphere. Colder temperatures are expected in higher
latitude regions, while warmer conditions are typically observed closer to the
equator.

In [ ]:
dec_1992 = monthly_avg.sel(time="1992-12-01")

plt.figure(figsize=(22,8))

dec_1992.plot(
    cmap="coolwarm",
    cbar_kwargs={"label": "Temperature (°C)"}
)

plt.title("Mean Air Temperature — December 1992")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.show()



---
### Time Series Analysis of Monthly Mean Temperature

To analyze temporal temperature variability, a time series of monthly mean
temperature is extracted for the location:

Latitude: **37.98° N**  
Longitude: **78.15° W**

The nearest grid cell in the dataset is selected to represent this location.

A **12-month rolling mean** is applied to smooth short-term seasonal
fluctuations and highlight longer-term trends in temperature.

Additionally, the **standard deviation of the monthly temperature values**
is calculated and visualized using a shaded region around the rolling mean,
which represents the variability in temperature over the study period.

In [ ]:
lat = 37.98
lon = -78.15  

# extract location and compute
loc = monthly_avg.sel(lat=lat, lon=lon, method="nearest").compute()

# rolling mean
rolling = loc.rolling(time=12, center=True).mean().dropna("time")

# standard deviation (constant)
std = loc.std()

# bounds
upper = rolling + std
lower = rolling - std

plt.figure(figsize=(22,8))

# monthly values
plt.plot(loc.time, loc, color="lightgray", label="Monthly Mean")

# rolling mean
plt.plot(
    rolling.time,
    rolling,
    color="red",
    linewidth=2,
    label="12-Month Rolling Mean"
)

# shading
plt.fill_between(
    rolling.time.values,
    lower.values,
    upper.values,
    color="red",
    alpha=0.1,
    label="±1 Std Dev"
)

plt.xlabel("Year")
plt.ylabel("Temperature (°C)")
plt.title("Monthly Mean Temperature (37.98°N, 78.15°W)")
plt.legend(loc="upper right")

plt.show()
